# 01 - Gaussian Process TabanlÄ± Bayes Optimizasyonu

Bu notebook'ta `GaussianProcessRegressor` bir **surrogate model** olarak kullanÄ±lÄ±r ve black-box minimizasyon iÃ§in tam Bayes optimizasyonu dÃ¶ngÃ¼sÃ¼ kurulur.

Ã–nemli terminoloji:

- `GaussianProcessRegressor`: regresyon modeli
- Bayes optimizasyonu: sequential optimizasyon yÃ¶ntemi
- acquisition / edinim fonksiyonu: sÄ±radaki deÄŸerlendirme noktasÄ±nÄ± seÃ§en Ã¶lÃ§Ã¼t

## 1. AlgoritmanÄ±n mantÄ±ÄŸÄ±

Bir minimizasyon problemi iÃ§in genel akÄ±ÅŸ:

1. BirkaÃ§ baÅŸlangÄ±Ã§ noktasÄ± deÄŸerlendir.
2. Bu gÃ¶zlemlerle GP surrogate modelini eÄŸit.
3. Her aday iÃ§in \(\mu(x)\) ve \(\sigma(x)\) tahmin et.
4. Bir acquisition function hesapla.
5. Acquisition function aÃ§Ä±sÄ±ndan en iyi yeni noktayÄ± seÃ§.
6. GerÃ§ek pahalÄ± amaÃ§ fonksiyonunu yalnÄ±zca o noktada Ã§alÄ±ÅŸtÄ±r.
7. GÃ¶zlemi veri kÃ¼mesine ekle.
8. BÃ¼tÃ§e bitene kadar devam et.

AmaÃ§, pahalÄ± gerÃ§ek fonksiyonu mÃ¼mkÃ¼n olduÄŸunca az Ã§aÄŸÄ±rmaktÄ±r.

## 2. Expected Improvement

Minimizasyon iÃ§in mevcut en iyi gÃ¶zlem \(f_{best}\) olsun.

\[
I(x) = \max(f_{best} - f(x) - \xi, 0)
\]

GP altÄ±nda \(f(x)\) belirsiz olduÄŸu iÃ§in doÄŸrudan iyileÅŸme yerine **beklenen iyileÅŸme** hesaplanÄ±r.

\[
EI(x)
=
(f_{best} - \mu(x) - \xi)\Phi(z)
+
\sigma(x)\phi(z)
\]

Burada:

\[
z =
\frac{f_{best} - \mu(x) - \xi}{\sigma(x)}
\]

\(\Phi\) standart normal daÄŸÄ±lÄ±mÄ±n CDF'i, \(\phi\) ise PDF'idir.

`xi` bÃ¼yÃ¼dÃ¼kÃ§e keÅŸif davranÄ±ÅŸÄ± artabilir.

## 3. LCB iÅŸaret konusu

Minimizasyon iÃ§in Lower Confidence Bound:

\[
LCB(x) = \mu(x) - \kappa\sigma(x)
\]

olarak tanÄ±mlanabilir ve **minimize edilir**.

Eski kodda Ã¶nemli bir hata ÅŸuydu: LCB hesaplandÄ±ktan sonra acquisition maximization dÃ¶ngÃ¼sÃ¼ne doÄŸrudan verilmiÅŸti. Bu iki yÃ¶n birbiriyle Ã§eliÅŸiyordu.

Bu repository'deki sÄ±nÄ±fta tÃ¼m edinim fonksiyonlarÄ± "bÃ¼yÃ¼k skor daha iyi" mantÄ±ÄŸÄ±na Ã§evrilmiÅŸtir. LCB iÃ§in:

\[
score(x) = -LCB(x)
\]

kullanÄ±lÄ±r.

In [ ]:
from pathlib import Path
import sys
import warnings

import numpy as np
import matplotlib.pyplot as plt

from sklearn.exceptions import ConvergenceWarning

# Notebook repository kÃ¶kÃ¼nden Ã§alÄ±ÅŸtÄ±rÄ±lmalÄ±dÄ±r.
PROJE_KOKU = Path.cwd()
if not (PROJE_KOKU / "src").exists() and (PROJE_KOKU.parent / "src").exists():
    PROJE_KOKU = PROJE_KOKU.parent

if not (PROJE_KOKU / "src").exists():
    raise FileNotFoundError(
        "Bu notebook'u repository kÃ¶k dizininden Ã§alÄ±ÅŸtÄ±rÄ±n. "
        "Ã–nce repository'yi klonlayÄ±p ilgili klasÃ¶re geÃ§in."
    )

sys.path.insert(0, str((PROJE_KOKU / "src").resolve()))

from gaussian_bo import GaussianProcessBayesOptimizer

# KÃ¼Ã§Ã¼k eÄŸitim Ã¶rneklerinde kernel optimizer sÄ±nÄ±r uyarÄ±larÄ± gÃ¶rÃ¼lebilir.
warnings.filterwarnings("ignore", category=ConvergenceWarning)

## 4. Basit black-box Ã¶rneÄŸi

AÅŸaÄŸÄ±daki fonksiyon analitik olarak elimizde olsa da onu optimizer aÃ§Ä±sÄ±ndan black-box kabul edeceÄŸiz.

Bu yalnÄ±zca algoritmayÄ± test etmek iÃ§indir.

In [ ]:
def amac_fonksiyonu(x):
    x0 = float(x[0])
    return (x0 - 2.0) ** 2 + 0.2 * np.sin(5.0 * x0)

optimizer = GaussianProcessBayesOptimizer(
    amac_fonksiyonu=amac_fonksiyonu,
    sinirlar=[[-2.0, 6.0]],
    baslangic_noktasi_sayisi=5,
    edinim_fonksiyonu="ei",
    xi=0.01,
    random_state=42,
)

sonuc = optimizer.optimize_et(
    iterasyon_sayisi=15,
    ayrintili=True,
)

print("En iyi x:", sonuc.en_iyi_x)
print("En iyi amaÃ§ deÄŸeri:", sonuc.en_iyi_y)

## 5. YakÄ±nsama grafiÄŸi

AÅŸaÄŸÄ±daki grafik, her gerÃ§ek amaÃ§ fonksiyonu deÄŸerlendirmesinden sonra o ana kadar bulunan en iyi deÄŸeri gÃ¶sterir.

GrafiÄŸin yataylaÅŸmasÄ± optimizer'Ä±n yeni ve daha iyi noktalar bulmakta zorlandÄ±ÄŸÄ±nÄ± gÃ¶sterebilir; fakat bu tek baÅŸÄ±na global optimumun bulunduÄŸu anlamÄ±na gelmez.

In [ ]:
kumulatif_en_iyi = np.minimum.accumulate(sonuc.y_gozlenen)

plt.figure(figsize=(9, 5))
plt.plot(
    np.arange(1, len(kumulatif_en_iyi) + 1),
    kumulatif_en_iyi,
    marker="o",
)
plt.xlabel("GerÃ§ek amaÃ§ fonksiyonu deÄŸerlendirme sayÄ±sÄ±")
plt.ylabel("O ana kadarki en iyi deÄŸer")
plt.title("Bayes optimizasyonu yakÄ±nsama grafiÄŸi")
plt.grid(True)
plt.show()

## 6. Son GP surrogate modelini Ã§izmek

1 boyutlu Ã¶rnekte GP'nin ne Ã¶ÄŸrendiÄŸini doÄŸrudan gÃ¶rebiliriz.

In [ ]:
X_grid = np.linspace(-2.0, 6.0, 500).reshape(-1, 1)
U_grid = optimizer._birim_araliga_donustur(X_grid)

optimizer.gp.fit(
    optimizer._birim_araliga_donustur(optimizer.X_gozlenen),
    optimizer.y_gozlenen,
)

mu, sigma = optimizer.gp.predict(U_grid, return_std=True)

plt.figure(figsize=(10, 5))
plt.plot(X_grid[:, 0], mu, label="GP tahmin ortalamasÄ±")
plt.fill_between(
    X_grid[:, 0],
    mu - 1.96 * sigma,
    mu + 1.96 * sigma,
    alpha=0.2,
    label="YaklaÅŸÄ±k %95 belirsizlik bandÄ±",
)
plt.scatter(
    optimizer.X_gozlenen[:, 0],
    optimizer.y_gozlenen,
    label="GerÃ§ek deÄŸerlendirmeler",
)
plt.xlabel("x")
plt.ylabel("AmaÃ§ deÄŸeri")
plt.title("Son Gaussian Process surrogate modeli")
plt.legend()
plt.grid(True)
plt.show()

## 7. Neden giriÅŸleri [0, 1] aralÄ±ÄŸÄ±na Ã¶lÃ§ekliyoruz?

MÃ¼hendislik problemlerinde karar deÄŸiÅŸkenlerin derinlikleri farkl°± olabilir:

- sÄ±caklÄ±k: 150â€“300
- basÄ±nc: 5â€“80
- sÃ¼re: 0.5â€“10
- devir: 1000â€“6000

GP'nin length-scale parametreleri bu Ã¶lÃ§eklerden doÄŸrudan etkilenir.

Repository'deki optimizer, karar deÄŸiÅŸkenlerini idahili olarak `[0, 1]` aralÄ±ÄŸÄ±na taÅŸÄ±r. AmaÃ§ fonksiyonuna ise gerÃ§ek mÃ¼hendislik birimleri gÃ¶nderilir.

## 8. GÃ¼rÃ¼ltÃ¼ hÃ¼mde amaÃ§ fonksiyonlarÄ±

GerÃ§ek fiziksel deney veya stokastik simÃ¼lasyon kullanÄ±lÄ±yorsa aynÄ± \(x\) iÃ§in farklÄ± \(y\) deÄŸerleri gÃ¶rÃ¼lebilir.

Bu durumda seÃ§enekler arasÄ±nda:

- aynÄ± noktayÄ± birden fazla kez deÄŸerlendirip ortalama almak,
- `alpha` ile bilinen/varsayÄ±lan gÃ¶zlem gÃ¼rÃ¼ltÃ¼sÃ¼nÃ¼ temsil etmek,
- kernel iÃ§inde `WhiteKernel` uygulamak,
- simÃ¼lasyon optimizasyonunda common random numbers kullanmak

bulunur.

Tek bir doÄŸru yaklaÅŸÄ±m yoktur. GÃ¼rÃ¼ltÃ¼nÃ¼n kaynaÄŸÄ±na gÃ¶re modelleme yapÄ±lmalÄ±dÄ±r.

## 9. `scikit-optimize` ile iliÅŸki

`scikit-optimize` ayrÄ± bir Python paketidir ve `gp_minimize` fonksiyonuyla benzer iÅŸi daha yÃ¼ksek seviyeli bir API Ã¼zerinden yapabilir.

Ã–rnek kullanÄ±m mantÄ±ÄŸÄ±:

```python
from skopt import gp_minimize
from skopt.space import Real

sonuc = gp_minimize(
    func=amac,
    dimensions=[Real(-2.0, 6.0)],
    n_calls=30,
    acq_func="EI",
    random_state=42,
)
```

Bu notebook'ta doÄŸrudan `GaussianProcessRegressor` tabanlÄ± kendi dÃ¶ngÃ¼mÃ¼zÃ¼ kullanmamÄ±zÄ±n nedeni, algoritmanÄ±n iÃ§ yapÄ±sÄ±nÄ± gÃ¶rÃ¼nÃ¼r kÄ±lmaktÄ±r.

## 10. Global optimum garantisi

Bayes optimizasyonu, genel black-box problemde global optimumu bulduÄŸunu pratikte garanti eden bir yÃ¶ntem olarak sunulmamalÄ±dÄ±r.

DoÄŸru raporlama dili:

- "en iyi bulunan Ã§Ã¶zÃ¼m"
- "deÄŸerlendirme bÃ¼tÃ§esi altÄ±nda bulunan Ã§Ã¶zÃ¼m"
- "referans aramaya gÃ¶re elde edilen sonuÃ§"

olmalÄ±dÄ±r.

KÃ¼Ã§Ã¼k problemlerde exhaustive search ile doÄŸrulama yapmak iyi bir eÄŸitim ve kalite kontrol yÃ¶ntemidir.